In [1]:
%matplotlib qt
import sys
import glob
import os
from pathlib import Path

src_path = Path.cwd().parent
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import time
import torch
import json
import gymnasium as gym
from envs.env import MiniGridEnvWrapper

from models.ppo_lstm import PPOLSTMAgent
from models.ride import RIDEAgent
from training.train import train_ppo_lstm, train_ride
from training.evaluate import evaluate_agent

/Users/qi/Desktop/CS238 Project/minigrid-with-curiosity/.venv/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [7]:
env_ids = [
    "MiniGrid-DoorKey-9x9-v0",
    "MiniGrid-SimpleCrossingS9N1-v0",
    "MiniGrid-LavaCrossingS9N2-v0",
    "MiniGrid-MultiRoom-N2-S4-v0",
    "MiniGrid-MultiRoom-N4-S5-v0",
    "MiniGrid-KeyCorridorS3R3-v0"
]
env = MiniGridEnvWrapper(env_ids[4], render_mode='rgb_array')

obs, info = env.reset()
print(f"Initial observation shape: {obs.shape}")
print(f"Action space: {env.action_space}")

num_steps = 100
for step in range(num_steps):
    action = env.action_space.sample()
    
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Step {step + 1}: Action={action}, Reward={reward}, Done={terminated or truncated}")
    
    env.display_interactive()
    
    if terminated or truncated:
        print(f"Episode finished after {step + 1} steps!")
        obs, info = env.reset()
        print("Environment reset for next episode")
        env.display_interactive()
    time.sleep(0.01)
env.close()

Initial observation shape: (7, 7, 3)
Action space: Discrete(7)
Step 1: Action=2, Reward=0, Done=False
Step 2: Action=2, Reward=0, Done=False
Step 3: Action=6, Reward=0, Done=False
Step 4: Action=1, Reward=0, Done=False
Step 5: Action=1, Reward=0, Done=False
Step 6: Action=4, Reward=0, Done=False
Step 7: Action=5, Reward=0, Done=False
Step 8: Action=4, Reward=0, Done=False
Step 9: Action=5, Reward=0, Done=False
Step 10: Action=1, Reward=0, Done=False
Step 11: Action=2, Reward=0, Done=False
Step 12: Action=6, Reward=0, Done=False
Step 13: Action=4, Reward=0, Done=False
Step 14: Action=5, Reward=0, Done=False
Step 15: Action=0, Reward=0, Done=False
Step 16: Action=4, Reward=0, Done=False
Step 17: Action=1, Reward=0, Done=False
Step 18: Action=1, Reward=0, Done=False
Step 19: Action=1, Reward=0, Done=False
Step 20: Action=5, Reward=0, Done=False
Step 21: Action=0, Reward=0, Done=False
Step 22: Action=3, Reward=0, Done=False
Step 23: Action=2, Reward=0, Done=False
Step 24: Action=2, Reward=

In [ ]:
agent = train_ppo_lstm(
        env=env,
        experiment_name="ppo_lstm_multiroom_n2",
        num_iterations=50,
        print_interval=1,  # Print every 10 iterations for cleaner output
        steps_per_iteration=2048,
        save_interval=50,
        device="cpu",  # or "cuda" or "mps"
        lr=3e-4,
        gamma=0.99,
        ppo_epochs=4,
        ppo_minibatch_size=4,
        hidden_size=256,
    )

In [ ]:
from models import ppo_lstm

agent_class = PPOLSTMAgent
env = MiniGridEnvWrapper(env_ids[3], render_mode='rgb_array')
ppo_lstm_runs = sorted(glob.glob("../runs/ppo_lstm_multiroom_n2_*"), key=os.path.getmtime, reverse=True)
if ppo_lstm_runs:
    latest_run = ppo_lstm_runs[0]
    config_path = os.path.join(latest_run, "config.json")
    checkpoint_path = os.path.join("../checkpoints", os.path.basename(latest_run), "final_model.pt")
    
    print(f"Using checkpoint: {checkpoint_path}")
    print(f"Using config: {config_path}")
    
    results = evaluate_agent(agent_class, env, checkpoint_path, config_path)


In [10]:
# Train RIDE agent
ride_agent = train_ride(
    env = env,
    experiment_name="ride_multiroom_n4",
    num_iterations=300,
    print_interval=1,
    steps_per_iteration=2048,
    save_interval=100,
    device="cpu",  # or "cuda" or "mps"
    lr=3e-4,  
    gamma=0.99,
    ppo_epochs=4,
    ppo_minibatch_size=8,
    hidden_size=256, 
    entropy_coef=0.01,  
    intrinsic_reward_coef=0.1, 
    use_intrinsic_normalization=True, 
)


Logging to: ../runs/ride_multiroom_n4_20251115_132627

Starting RIDE training: ride_multiroom_n4_20251115_132627
Intrinsic reward coefficient: 0.1

[   1/300] Frames:   2,048 | Reward:    0.00 | Intrinsic:  2.187 | Length:  120.0 | Loss: 0.0089/0.0017 | FD: 0.0489 ID: 1.9458
[   2/300] Frames:   4,096 | Reward:    0.00 | Intrinsic:  2.255 | Length:  120.0 | Loss: -0.0484/0.0016 | FD: 0.0377 ID: 1.9450
[   3/300] Frames:   6,144 | Reward:    0.00 | Intrinsic:  1.836 | Length:  120.0 | Loss: -0.0033/0.0007 | FD: 0.0278 ID: 1.9415
[   4/300] Frames:   8,192 | Reward:    0.00 | Intrinsic:  1.619 | Length:  120.0 | Loss: 0.0104/0.0006 | FD: 0.0228 ID: 1.9291
[   5/300] Frames:  10,240 | Reward:    0.00 | Intrinsic:  1.382 | Length:  120.0 | Loss: -0.0138/0.0006 | FD: 0.0199 ID: 1.8970
[   6/300] Frames:  12,288 | Reward:    0.00 | Intrinsic:  1.222 | Length:  120.0 | Loss: -0.0049/0.0006 | FD: 0.0204 ID: 1.8698
[   7/300] Frames:  14,336 | Reward:    0.00 | Intrinsic:  0.964 | Length:  120.

KeyboardInterrupt: 

In [6]:
# Evaluate RIDE agent
ride_agent_class = RIDEAgent
env_ride_eval = MiniGridEnvWrapper(env_ids[3], render_mode='rgb_array')

ride_runs = sorted(glob.glob("../runs/ride_multiroom_n2_*"), key=os.path.getmtime, reverse=True)
if ride_runs:
    latest_run = ride_runs[0]
    config_path = os.path.join(latest_run, "config.json")
    checkpoint_path = os.path.join("../checkpoints", os.path.basename(latest_run), "final_model.pt")
    
    print(f"Using checkpoint: {checkpoint_path}")
    print(f"Using config: {config_path}")
    
    results = evaluate_agent(ride_agent_class, env_ride_eval, checkpoint_path, config_path)
else:
    print("No RIDE training runs found. Please train the agent first.")


Using checkpoint: ../checkpoints/ride_multiroom_n2_20251115_131706/final_model.pt
Using config: ../runs/ride_multiroom_n2_20251115_131706/config.json
Loading checkpoint from ../checkpoints/ride_multiroom_n2_20251115_131706/final_model.pt...
Checkpoint loaded successfully!

Running 10 evaluation episodes...
Episode  1/10 | Steps:   7 | Reward:   0.84
Episode  2/10 | Steps:  12 | Reward:   0.73
Episode  3/10 | Steps:  10 | Reward:   0.78
Episode  4/10 | Steps:   8 | Reward:   0.82
Episode  5/10 | Steps:  11 | Reward:   0.75
Episode  6/10 | Steps:   8 | Reward:   0.82
Episode  7/10 | Steps:   7 | Reward:   0.84
Episode  8/10 | Steps:   5 | Reward:   0.89
Episode  9/10 | Steps:   7 | Reward:   0.84
Episode 10/10 | Steps:  10 | Reward:   0.78

Evaluation Summary:
  Episodes:     10
  Mean Reward:  0.81 ± 0.05
  Min/Max:      0.73 / 0.89
  Mean Length:  8.50 ± 2.06
  Success Rate: 100.0%
